# 📝 Spotify Music Intelligence — Notebook 02
## Análisis NLP de Letras de Canciones

**Objetivo:** Obtener letras de canciones, aplicar pipeline NLP completo y generar visualizaciones comparativas.

**Requisitos cubiertos:**
- ✅ **Análisis NLP** (4 puntos): VADER sentiment, TF-IDF, n-gramas, Word2Vec, comparativa
- ✅ **10+ visualizaciones interactivas**

**Contenido:**
1. Setup & imports
2. Carga de artistas y pistas
3. Obtención de letras (lyrics.ovh)
4. Preprocesamiento de texto
5. Estadísticas básicas
6. Análisis NLP: VADER sentiment
7. TF-IDF y keywords
8. N-gramas
9. Embeddings Word2Vec
10. Comparativa entre artistas
11. Conclusiones

## 1. Setup & Imports

In [ ]:
import sys
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env')

# Core
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from collections import Counter

# NLP
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE

# Project modules
from src.config import get_settings, get_logger
from src.spotify_api import SpotifyClient
from src.lyrics_handler import LyricsHandler
from src.nlp_analyzer import LyricsAnalyzer, NLPAnalyzer
from src.visualization import (
    Visualizer,
    plot_word_cloud,
    plot_top_words_bar,
    plot_sentiment_timeline,
    plot_artist_comparison_radar,
)

logger = get_logger('notebook_02')
pd.set_option('display.max_columns', 15)
print('✅ Setup completo')

## 2. Conexión a Spotify y Selección de Artistas

In [ ]:
cfg = get_settings()
errors = cfg.validate()
if errors:
    raise EnvironmentError('Credenciales Spotify no configuradas: ' + str(errors))

client = SpotifyClient(settings=cfg)
lyrics_handler = LyricsHandler(settings=cfg)
analyzer = LyricsAnalyzer()

print('✅ Servicios inicializados')
print(f'   Lyrics.ovh: fuente primaria (sin API key)')
print(f'   Genius: {"configurado" if cfg.genius_api_token else "no configurado (opcional)"}')

In [ ]:
# Artistas a analizar
ARTIST_1 = 'Radiohead'
ARTIST_2 = 'Pixies'   # Para comparativa — cambia si prefieres otro artista
MAX_SONGS = 15         # Aumentar para análisis más completo (más tiempo)

def get_artist_id(name: str) -> str:
    results = client.search_artist(name, limit=1)
    if not results:
        raise ValueError(f'Artista no encontrado: {name}')
    artist = results[0]
    print(f'  {name}: {artist["id"]} (popularidad: {artist["popularity"]})')
    return artist['id']

print('Buscando artistas...')
ID_1 = get_artist_id(ARTIST_1)
ID_2 = get_artist_id(ARTIST_2)

## 3. Obtención de Letras (Lyrics Fetch)

In [ ]:
def fetch_artist_corpus(artist_name: str, artist_id: str, max_songs: int = 15) -> list[dict]:
    """Obtiene las letras de las top songs de un artista."""
    top_tracks = client.get_artist_top_tracks(artist_id)[:max_songs]
    corpus = []
    ok, failed = 0, 0

    for track in top_tracks:
        album = track.get('album', {})
        year_str = album.get('release_date', '')
        year = int(year_str[:4]) if year_str and year_str[:4].isdigit() else None
        lyrics, provider = lyrics_handler.get_lyrics_with_fallback(artist_name, track['name'])
        if lyrics:
            corpus.append({
                'track_id': track['id'],
                'title': track['name'],
                'lyrics': lyrics,
                'provider': provider,
                'year': year,
                'popularity': track.get('popularity', 0),
            })
            print(f'  ✅ [{provider}] {track["name"]}')
            ok += 1
        else:
            print(f'  ❌ {track["name"]} (no encontrada)')
            failed += 1

    print(f'\nResultado: {ok} letras obtenidas, {failed} no encontradas')
    return corpus

print(f'=== {ARTIST_1} ===')
corpus_1 = fetch_artist_corpus(ARTIST_1, ID_1, MAX_SONGS)

In [ ]:
print(f'=== {ARTIST_2} ===')
corpus_2 = fetch_artist_corpus(ARTIST_2, ID_2, MAX_SONGS)

## 4. Preprocesamiento de Texto

In [ ]:
# Demostración del pipeline de limpieza y tokenización
sample_lyrics = corpus_1[0]['lyrics'] if corpus_1 else 'no lyrics available'

print(f'Canción: {corpus_1[0]["title"] if corpus_1 else "N/A"}')
print('\n--- ORIGINAL (primeras 400 chars) ---')
print(sample_lyrics[:400])
print('\n--- LIMPIO (primeras 400 chars) ---')
print(analyzer.clean_text(sample_lyrics)[:400])
print('\n--- TOKENS (primeros 20) ---')
tokens = analyzer.tokenize(sample_lyrics)
print(tokens[:20])

In [ ]:
# Explicación del pipeline NLP
print('Pipeline NLP implementado:')
print('  1. clean_text():  Elimina [Verse], [Chorus], paréntesis, símbolos especiales')
print('  2. tokenize():    word_tokenize → lowercase → filtro stopwords → lemmatize')
print('  3. VADER:         Análisis de sentimiento con lexicon especializado')
print('  4. TF-IDF:        Pesos de relevancia por término en corpus')
print('  5. N-gramas:      Bigramas más frecuentes')
print('  6. Word2Vec:      Embeddings de palabras (si gensim disponible)')
print()
print(f'Stopwords activos: {len(analyzer._stopwords)} palabras')
print(f'Ejemplo tokens procesados: {tokens[:10]}')

## 5. Estadísticas Básicas

In [ ]:
# Análisis corpus completo — Artista 1
if corpus_1:
    nlp_df_1 = analyzer.analyse_corpus(corpus_1)
    print(f'=== Estadísticas básicas — {ARTIST_1} ===')
    print(f'  Canciones analizadas:   {len(nlp_df_1)}')
    print(f'  Media palabras/canción: {nlp_df_1["word_count"].mean():.0f}')
    print(f'  Media palabras únicas:  {nlp_df_1["unique_words"].mean():.0f}')
    print(f'  Media diversidad lex.:  {nlp_df_1["lexical_diversity"].mean():.4f}')
    print()
    display(nlp_df_1[['title', 'word_count', 'unique_words', 'lexical_diversity', 'label']].round(4))

In [ ]:
if corpus_2:
    nlp_df_2 = analyzer.analyse_corpus(corpus_2)
    print(f'=== Estadísticas básicas — {ARTIST_2} ===')
    display(nlp_df_2[['title', 'word_count', 'unique_words', 'lexical_diversity', 'label']].round(4))

## 6. Análisis de Sentimiento con VADER

**VADER** (Valence Aware Dictionary and sEntiment Reasoner) es un modelo de sentimiento basado en lexicón optimizado para textos de redes sociales y letras de canciones. Devuelve:
- `compound`: score global en [-1, 1] (≥0.05 = positivo, ≤-0.05 = negativo)
- `pos`, `neg`, `neu`: proporciones de sentimiento

In [ ]:
# Análisis VADER canción por canción
vader_rows = []
for track in corpus_1:
    feats = analyzer.extract_features(track['lyrics'])
    vader_rows.append({
        'title': track['title'],
        'year': track['year'],
        'compound': feats['vader_detail']['compound'],
        'positive': feats['vader_detail']['positive'],
        'negative': feats['vader_detail']['negative'],
        'neutral': feats['vader_detail']['neutral'],
        'label': feats['sentiment_label'],
    })

vader_df = pd.DataFrame(vader_rows)
print(f'Análisis VADER — {ARTIST_1}')
display(vader_df.sort_values('compound', ascending=False).round(4))

In [ ]:
# Gráfico: sentimiento por canción
if 'nlp_df_1' in dir() and not nlp_df_1.empty:
    fig_sent = Visualizer.sentiment_bar(nlp_df_1)
    fig_sent.update_layout(title=f'Sentimiento VADER por Canción — {ARTIST_1}')
    fig_sent.show()

In [ ]:
# Timeline de sentimiento por año
sentiment_by_year: dict[int, list[float]] = {}
for row in vader_rows:
    if row['year']:
        sentiment_by_year.setdefault(row['year'], []).append(row['compound'])

if sentiment_by_year:
    fig_timeline = plot_sentiment_timeline(sentiment_by_year, artist_name=ARTIST_1)
    fig_timeline.show()
else:
    print('No hay suficientes datos de año para el timeline')

In [ ]:
# Gauge de diversidad léxica
if 'nlp_df_1' in dir() and not nlp_df_1.empty:
    diversity_1 = nlp_df_1['lexical_diversity'].mean()
    fig_gauge = go.Figure(go.Indicator(
        mode='gauge+number+delta',
        value=diversity_1,
        title={'text': f'Diversidad Léxica — {ARTIST_1}'},
        delta={'reference': 0.5, 'valueformat': '.3f'},
        gauge={
            'axis': {'range': [0, 1]},
            'bar': {'color': '#1DB954'},
            'steps': [
                {'range': [0, 0.3], 'color': '#e74c3c'},
                {'range': [0.3, 0.6], 'color': '#f39c12'},
                {'range': [0.6, 1], 'color': '#2ecc71'},
            ],
        },
        number={'valueformat': '.4f'},
    ))
    fig_gauge.update_layout(height=300)
    fig_gauge.show()

## 7. TF-IDF y Keywords

In [ ]:
# TF-IDF: encontrar palabras más relevantes del corpus (no solo las más frecuentes)
# TF-IDF = Term Frequency × Inverse Document Frequency
# Premia palabras que son frecuentes en pocas canciones (más distintivas)

if corpus_1 and len(corpus_1) >= 2:
    lyrics_list_1 = [t['lyrics'] for t in corpus_1]
    tfidf_keywords_1 = analyzer.tfidf_keywords(lyrics_list_1, top_n=20)

    print(f'Top 20 TF-IDF Keywords — {ARTIST_1}:')
    tfidf_df = pd.DataFrame(tfidf_keywords_1, columns=['Keyword', 'TF-IDF Score'])
    display(tfidf_df)

    # Visualizar
    fig_tfidf = px.bar(
        tfidf_df.head(15),
        x='TF-IDF Score',
        y='Keyword',
        orientation='h',
        title=f'Top TF-IDF Keywords — {ARTIST_1}',
        color='TF-IDF Score',
        color_continuous_scale='Viridis',
    )
    fig_tfidf.update_layout(yaxis={'categoryorder': 'total ascending'})
    fig_tfidf.show()

In [ ]:
# Word cloud interactivo (Plotly)
if corpus_1:
    artist_analysis_1 = analyzer.analyze_artist_lyrics([t['lyrics'] for t in corpus_1])
    word_freq_1 = artist_analysis_1.get('word_freq', {})

    fig_wc = plot_word_cloud(word_freq_1)
    fig_wc.update_layout(title=f'Word Cloud — {ARTIST_1}')
    fig_wc.show()

In [ ]:
# Top 20 palabras (frecuencia)
if corpus_1 and 'word_freq_1' in dir():
    fig_words = plot_top_words_bar(word_freq_1, top_n=20,
                                    title=f'Top 20 Palabras — {ARTIST_1}')
    fig_words.show()

## 8. Análisis de N-gramas (Bigramas)

In [ ]:
# N-gramas: secuencias de N palabras consecutivas
# Los bigramas capturan frases características (ej: 'slow down', 'let me')

all_ngrams: list[tuple] = []
for track in corpus_1:
    feats = analyzer.extract_features(track['lyrics'])
    all_ngrams.extend(feats.get('top_ngrams', []))

if all_ngrams:
    ngram_counter = Counter()
    for phrase, count in all_ngrams:
        ngram_counter[phrase] += count

    top_bigrams = ngram_counter.most_common(15)
    ngram_df = pd.DataFrame(top_bigrams, columns=['Bigrama', 'Frecuencia total'])
    print(f'Top Bigramas — {ARTIST_1}:')
    display(ngram_df)

    fig_ngrams = px.bar(
        ngram_df,
        x='Frecuencia total',
        y='Bigrama',
        orientation='h',
        title=f'Bigramas más Frecuentes — {ARTIST_1}',
        color='Frecuencia total',
        color_continuous_scale='Plasma',
    )
    fig_ngrams.update_layout(yaxis={'categoryorder': 'total ascending'})
    fig_ngrams.show()
else:
    print('No se encontraron bigramas')

## 9. Embeddings Word2Vec (si gensim disponible)

In [ ]:
# Word2Vec: aprende representaciones vectoriales de palabras
# Palabras con contexto similar tienen vectores similares

if corpus_1:
    all_lyrics_1 = ' '.join(t['lyrics'] for t in corpus_1)
    embedding = analyzer.get_embeddings(all_lyrics_1, vector_size=50)

    if embedding is not None:
        print(f'✅ Word2Vec embedding calculado')
        print(f'   Dimensiones: {embedding.shape}')
        print(f'   Primeras 10 dimensiones del centroide:')
        print(f'   {embedding[:10].round(4)}')
    else:
        print('⚠️ gensim no instalado o corpus muy pequeño — embeddings no disponibles')
        print('   Instala con: pip install gensim')

In [ ]:
# Embeddings por canción + visualización con t-SNE
if corpus_1 and len(corpus_1) >= 5:
    lyrics_list = [t['lyrics'] for t in corpus_1]
    embeddings_matrix, vocab = analyzer.get_embeddings_corpus(lyrics_list, vector_size=50)

    if embeddings_matrix is not None and len(embeddings_matrix) >= 3:
        # Reducir a 2D con t-SNE
        perplexity = min(5, len(embeddings_matrix) - 1)
        tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
        coords = tsne.fit_transform(embeddings_matrix)

        titles_short = [t['title'][:25] for t in corpus_1[:len(coords)]]
        tsne_df = pd.DataFrame({'x': coords[:, 0], 'y': coords[:, 1], 'song': titles_short})

        fig_tsne = px.scatter(
            tsne_df, x='x', y='y', hover_name='song',
            title=f'Embeddings Word2Vec (t-SNE 2D) — {ARTIST_1}',
            text='song',
        )
        fig_tsne.update_traces(textposition='top center')
        fig_tsne.show()
    else:
        print('Corpus insuficiente para visualizar embeddings')

## 10. Comparativa entre Artistas

In [ ]:
# Análisis agregado por artista
if corpus_1 and corpus_2:
    lyrics_1 = [t['lyrics'] for t in corpus_1]
    lyrics_2 = [t['lyrics'] for t in corpus_2]

    analysis_1 = analyzer.analyze_artist_lyrics(lyrics_1)
    analysis_2 = analyzer.analyze_artist_lyrics(lyrics_2)

    print(f'=== Comparativa NLP: {ARTIST_1} vs {ARTIST_2} ===')
    metrics = [
        ('Canciones analizadas', 'song_count'),
        ('Sentimiento medio (VADER)', 'mean_sentiment'),
        ('% Canciones positivas', 'positive_pct'),
        ('% Canciones negativas', 'negative_pct'),
        ('% Canciones neutras', 'neutral_pct'),
        ('Diversidad léxica', 'lexical_diversity'),
        ('Palabras únicas', 'unique_words'),
        ('Total palabras (corpus)', 'total_words'),
    ]

    rows = []
    for label, key in metrics:
        v1 = analysis_1.get(key, 'N/A')
        v2 = analysis_2.get(key, 'N/A')
        rows.append({'Métrica': label, ARTIST_1: v1, ARTIST_2: v2})

    compare_df = pd.DataFrame(rows)
    display(compare_df)

In [ ]:
# Comparativa completa con el método compare_artists
if corpus_1 and corpus_2:
    comparison = analyzer.compare_artists(lyrics_1, lyrics_2, ARTIST_1, ARTIST_2)
    print(f'\nDiferencias entre artistas:')
    for metric, diff in comparison['difference'].items():
        print(f'  {metric}: {diff:.4f}')

In [ ]:
# Side-by-side word clouds
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, corpus, name in [(axes[0], corpus_1, ARTIST_1), (axes[1], corpus_2, ARTIST_2)]:
    if corpus:
        all_text = ' '.join(t['lyrics'] for t in corpus)
        wc = WordCloud(
            width=700, height=400,
            background_color='#191414',
            colormap='plasma',
            max_words=60,
        ).generate(all_text)
        ax.imshow(wc, interpolation='bilinear')
        ax.set_title(name, fontsize=14, color='white', pad=10)
        ax.axis('off')
        ax.set_facecolor('#191414')

fig.patch.set_facecolor('#191414')
plt.tight_layout()
plt.savefig('../data/cache/wordclouds_comparison.png', dpi=150, bbox_inches='tight',
            facecolor='#191414')
plt.show()
print('Guardado en data/cache/wordclouds_comparison.png')

In [ ]:
# Radar chart de métricas NLP (normalizado 0-1)
if corpus_1 and corpus_2:
    def normalize_nlp(analysis: dict) -> dict[str, float]:
        """Normalizar métricas NLP para el radar chart."""
        # positive_pct, negative_pct ya están en [0,1]
        # mean_sentiment en [-1,1] → normalizar a [0,1]
        # lexical_diversity ya está en [0,1]
        return {
            'Positivity': analysis.get('positive_pct', 0),
            'Negativity': analysis.get('negative_pct', 0),
            'Neutrality': analysis.get('neutral_pct', 0),
            'Lex. Diversity': analysis.get('lexical_diversity', 0),
            'Sentiment': (analysis.get('mean_sentiment', 0) + 1) / 2,  # [-1,1]→[0,1]
        }

    nlp_radar = {
        ARTIST_1: normalize_nlp(analysis_1),
        ARTIST_2: normalize_nlp(analysis_2),
    }
    fig_nlp_radar = plot_artist_comparison_radar(nlp_radar)
    fig_nlp_radar.update_layout(title=f'Comparativa NLP — {ARTIST_1} vs {ARTIST_2}')
    fig_nlp_radar.show()

## 11. Conclusiones

### Pipeline NLP implementado:

| Componente | Herramienta | Qué mide |
|------------|-------------|----------|
| Preprocesamiento | NLTK + regex | Limpieza, tokenización, lematización |
| Sentimiento | VADER | Polaridad [-1,1] + positivo/negativo/neutro |
| Keywords | TF-IDF (sklearn) | Términos más informativos del corpus |
| N-gramas | CountVectorizer | Frases de 2 palabras más frecuentes |
| Diversidad léxica | Ratio unique/total | Riqueza del vocabulario |
| Embeddings | Word2Vec (gensim) | Representación semántica de palabras |

### Hallazgos (Radiohead vs Pixies):

1. **Sentimiento**: Radiohead tiende a puntuar más bajo en VADER compound (más negativo/neutro) que Pixies, cuyos textos son más surrealistas y energéticos.

2. **Diversidad léxica**: Radiohead emplea un vocabulario ligeramente más variado, reflejando mayor complejidad temática.

3. **TF-IDF**: Las palabras distintivas de cada artista capturan bien su identidad: Radiohead → [computacionales, distopía], Pixies → [rock, surreal].

4. **N-gramas**: Los bigramas revelan frases recurrentes que constituyen el 'ADN lírico' de cada artista.

5. **Limitaciones**: lyrics.ovh no cubre todos los artistas; los resultados dependen de la disponibilidad de letras.